# Assess whether profiles persist over time

**Data and method:** the national benchmark and its algorithm, feature and telephone sensitivity boundaries. The temporal calculation is parallel to the telephone calculations, not computationally downstream from them.

**Purpose:** inspect checksum-verified half-year and quarterly authority tables and decide whether shorter windows support or replace the annual model.

This notebook validates included authority tables. It does not claim to reconstruct the full monthly national feature pipeline included in the dissertation analysis.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_PATH = ROOT / 'configs' / 'reference_apr2025_mar2026.json'
from gpap2.config import load_config
REFERENCE_CONFIG = load_config(CONFIG_PATH)
AUTHORITY_MANIFEST = REFERENCE_CONFIG.resolve(REFERENCE_CONFIG.authority_checksum_file)
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 6)


In [2]:
import pandas as pd
from gpap2.io import validate_authority_file

canonical_path = ROOT / 'outputs' / 'tables' / 'temporal_canonical_period_metrics.csv'
structural_path = ROOT / 'outputs' / 'tables' / 'temporal_structural_period_metrics.csv'
validate_authority_file(canonical_path, AUTHORITY_MANIFEST)
validate_authority_file(structural_path, AUTHORITY_MANIFEST)
canonical = pd.read_csv(canonical_path)
structural = pd.read_csv(structural_path)
canonical[['period_type', 'period_name', 'practices', 'annual_agreement_share', 'adjusted_rand_index_vs_annual', 'silhouette_score']]

,period_type,period_name,practices,annual_agreement_share,adjusted_rand_index_vs_annual,silhouette_score
0,half_year,H1_Apr_Sep_2025,5924,0.872046,0.652665,0.109059
1,half_year,H2_Oct_2025_Mar_2026,5924,0.921337,0.778453,0.120163
2,quarter,Q1_Apr_Jun_2025,5677,0.733310,0.357795,0.080075
3,quarter,Q2_Jul_Sep_2025,5677,0.869297,0.646924,0.112129
4,quarter,Q3_Oct_Dec_2025,5677,0.893606,0.708947,0.113552
5,quarter,Q4_Jan_Mar_2026,5677,0.833891,0.563249,0.103475


In [3]:
summary = canonical.groupby('period_type').agg(
    periods=('period_name', 'count'),
    minimum_practices=('practices', 'min'),
    median_annual_agreement=('annual_agreement_share', 'median'),
    median_ari_vs_annual=('adjusted_rand_index_vs_annual', 'median'),
).reset_index()
assert set(summary['period_type']) == {'half_year', 'quarter'}
assert len(structural) == len(canonical) == 6
summary

,period_type,periods,minimum_practices,median_annual_agreement,median_ari_vs_annual
0,half_year,2,5924,0.896691,0.715559
1,quarter,4,5677,0.851594,0.605086


## Decision

Half-year and quarterly partitions retain related structure but show genuine within-year reassignment. The April 2025 to March 2026 annual model remains the reference; temporal results are sensitivity evidence rather than replacement profiles.

**What this establishes:** the robustness evidence bounds the external-context interpretation of the fixed national profiles.